# Численное моделирование обтекания цилиндра сильно неявным методом (SIP)
В данной работе решается задача обтекания круглого цилиндра идеальным сжимаемым газом с использованием потенциальной постановки и метода Стоуна (SIP).

## Математическая модель
Уравнение неразрывности для потенциала скорости $\varphi$ ($\vec{v} = \nabla \varphi$):
$$\frac{\partial}{\partial \xi} (\rho \varphi_\xi) + \frac{\partial}{\partial \eta} (\rho \varphi_\eta) = 0$$

В криволинейных координатах $(\xi, \eta)$, связанных с декартовыми как $x = e^\xi \cos \eta, y = e^\xi \sin \eta$:
$$\frac{\partial}{\partial \xi} (\rho \varphi_\xi) + \frac{\partial}{\partial \eta} (\rho \varphi_\eta) = 0$$

## Метод SIP (Strongly Implicit Procedure)
Для применения метода ПНМ (SIP) уравнение записывается в общем пятиточечном виде для поправки $\delta \varphi = \varphi^{(k+1)} - \varphi^{(k)}$:
$$\hat{A}_{i,j} \delta \varphi_{i,j-1} + A_{i,j} \delta \varphi_{i-1,j} + B_{i,j} \delta \varphi_{i,j} + C_{i,j} \delta \varphi_{i+1,j} + \hat{C}_{i,j} \delta \varphi_{i,j+1} = -F_{i,j}$$

где $-F_{i,j}$ — невязка на текущей итерации. Коэффициенты определяются выражениями:
- $A_{i,j} = \rho_{i-1/2,j} / \Delta \xi^2$ (Запад)
- $C_{i,j} = \rho_{i+1/2,j} / \Delta \xi^2$ (Восток)
- $\hat{A}_{i,j} = \rho_{i,j-1/2} / \Delta \eta^2$ (Юг)
- $\hat{C}_{i,j} = \rho_{i,j+1/2} / \Delta \eta^2$ (Север)
- $B_{i,j} = -(A_{i,j} + C_{i,j} + \hat{A}_{i,j} + \hat{C}_{i,j})$ (Центр)


In [12]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Параметры задачи (синхронизировано с SLOR)
gamma = 1.4
xi_max = 4.0
N_xi = 100
N_eta = 120
h_xi = xi_max / N_xi
h_eta = 2 * np.pi / N_eta

eps = 1e-5
alpha_stone = 0.92  # Параметр SIP

xi = np.linspace(0, xi_max, N_xi + 1)
eta = np.linspace(0, 2 * np.pi, N_eta + 1)
XI, ETA = np.meshgrid(xi, eta, indexing='ij')

In [13]:
import numpy as np
def get_density(phi, M_inf):
    if M_inf == 0: return np.ones_like(phi)
    phi_xi = np.zeros_like(phi)
    phi_eta = np.zeros_like(phi)
    
    # Центральные разности для внутренних узлов
    phi_xi[1:-1, :] = (phi[2:, :] - phi[:-2, :]) / (2 * h_xi)
    phi_xi[0, :] = 0  # Условие непротекания на поверхности
    phi_xi[-1, :] = (phi[-1, :] - phi[-2, :]) / h_xi  # Разность назад на внешней границе
    
    phi_eta[:, 1:-1] = (phi[:, 2:] - phi[:, :-2]) / (2 * h_eta)
    # Симметрия на осях
    phi_eta[:, 0] = 0
    phi_eta[:, -1] = 0
    
    exp_2xi = np.exp(-2 * XI)
    q2 = exp_2xi * (phi_xi**2 + phi_eta**2)
    # Ограничение для дозвукового режима (стабилизация)
    q2_crit = 2/((gamma-1)*M_inf**2) + 1.0
    q2 = np.clip(q2, 0, q2_crit)
    
    rho = (1 + 0.5 * (gamma - 1) * M_inf**2 * (1 - q2))**(1/(gamma-1))
    return rho

In [14]:
class StoneSolver:
    def __init__(self, nx, ny):
        self.nx, self.ny = nx, ny
        shape = (nx, ny)
        self.b = np.zeros(shape)
        self.c = np.zeros(shape)
        self.d = np.zeros(shape)
        self.e = np.zeros(shape)
        self.f = np.zeros(shape)

    def factorize(self, S, W, B, E, N, alpha):
        # S=hat_A, W=A, B=B, E=C, N=hat_C
        for j in range(self.ny):
            for i in range(self.nx):
                b_prev = self.f[i, j-1] if j > 0 else 0
                e_prev = self.e[i, j-1] if j > 0 else 0
                c_prev = self.e[i-1, j] if i > 0 else 0
                f_prev = self.f[i-1, j] if i > 0 else 0

                self.b[i, j] = S[i, j] / (1.0 + alpha * e_prev)
                self.c[i, j] = W[i, j] / (1.0 + alpha * f_prev)
                
                self.d[i, j] = B[i, j] + alpha * (self.b[i, j] * e_prev + self.c[i, j] * f_prev) \
                               - self.b[i, j] * b_prev - self.c[i, j] * c_prev
                
                self.e[i, j] = (E[i, j] - alpha * self.c[i, j] * f_prev) / self.d[i, j]
                self.f[i, j] = (N[i, j] - alpha * self.b[i, j] * e_prev) / self.d[i, j]
                
    def solve(self, res):
        y = np.zeros((self.nx, self.ny))
        # Прямой ход
        for j in range(self.ny):
            for i in range(self.nx):
                y_s = y[i, j-1] if j > 0 else 0
                y_w = y[i-1, j] if i > 0 else 0
                y[i, j] = (res[i, j] - self.b[i, j] * y_s - self.c[i, j] * y_w) / self.d[i, j]
        
        dx = np.zeros((self.nx, self.ny))
        # Обратный ход
        for j in range(self.ny-1, -1, -1):
            for i in range(self.nx-1, -1, -1):
                dx_e = dx[i+1, j] if i < self.nx-1 else 0
                dx_n = dx[i, j+1] if j < self.ny-1 else 0
                dx[i, j] = y[i, j] - self.e[i, j] * dx_e - self.f[i, j] * dx_n
        return dx

def solve_for_mach_sip(M_inf, phi_init=None):
    # Инициализация
    if phi_init is None:
        phi = (np.exp(XI) + np.exp(-XI)) * np.cos(ETA)
    else:
        phi = phi_init.copy()
        
    solver = StoneSolver(N_xi + 1, N_eta + 1)
    
    for it in range(2000):
        rho = get_density(phi, M_inf)
        
        # Коэффициенты внутренних узлов и полуцелые плотности
        rho_e = 0.5 * (rho[1:, :] + rho[:-1, :])
        rho_w = 0.5 * (rho[:-1, :] + rho[1:, :]) # same but shifted indices needed
        # Для удобства создадим расширенные массивы плотности на гранях
        r_e = np.zeros((N_xi + 1, N_eta + 1))
        r_w = np.zeros((N_xi + 1, N_eta + 1))
        r_n = np.zeros((N_xi + 1, N_eta + 1))
        r_s = np.zeros((N_xi + 1, N_eta + 1))
        
        r_e[:-1, :] = 0.5 * (rho[:-1, :] + rho[1:, :])
        r_w[1:, :] = 0.5 * (rho[1:, :] + rho[:-1, :])
        r_n[:, :-1] = 0.5 * (rho[:, :-1] + rho[:, 1:])
        r_s[:, 1:] = 0.5 * (rho[:, 1:] + rho[:, :-1])
        
        # Пятиточечные коэффициенты
        W = r_w / h_xi**2
        E = r_e / h_xi**2
        S = r_s / h_eta**2
        N = r_n / h_eta**2
        B = -(W + E + S + N)
        
        # Невязка F
        # F = (rho_e*(phi_e - phi) - rho_w*(phi - phi_w))/hxi^2 + ...
        F = np.zeros_like(phi)
        F[1:-1, 1:-1] = (r_e[1:-1, 1:-1]*(phi[2:, 1:-1] - phi[1:-1, 1:-1]) - \
                         r_w[1:-1, 1:-1]*(phi[1:-1, 1:-1] - phi[:-2, 1:-1])) / h_xi**2 + \
                        (r_n[1:-1, 1:-1]*(phi[1:-1, 2:] - phi[1:-1, 1:-1]) - \
                         r_s[1:-1, 1:-1]*(phi[1:-1, 1:-1] - phi[1:-1, :-2])) / h_eta**2
        
        # Граничные условия в матрице (согласно 5_pnm.tex)
        # 1. Поверхность (i=0)
        # Условие phi_{-1} = phi_1 => B_0 = B_0 + A_0 (но здесь мы используем r_w=0 и удваиваем r_e)
        W[0, :] = 0
        E[0, :] = 2 * r_e[0, :] / h_xi**2
        B[0, :] = -(E[0, :] + S[0, :] + N[0, :])
        F[0, 1:-1] = (r_e[0, 1:-1]*(phi[1, 1:-1] - phi[0, 1:-1]) * 2) / h_xi**2 + \
                     (r_n[0, 1:-1]*(phi[0, 2:] - phi[0, 1:-1]) - r_s[0, 1:-1]*(phi[0, 1:-1] - phi[0, :-2])) / h_eta**2
        
        # 2. Симметрия (j=0 и j=N_eta)
        S[:, 0] = 0
        N[:, 0] = 2 * r_n[:, 0] / h_eta**2
        B[:, 0] = -(W[:, 0] + E[:, 0] + N[:, 0])
        F[1:-1, 0] = (r_e[1:-1, 0]*(phi[2:, 0] - phi[1:-1, 0]) - r_w[1:-1, 0]*(phi[1:-1, 0] - phi[:-2, 0])) / h_xi**2 + \
                      (r_n[1:-1, 0]*(phi[1:-1, 1] - phi[1:-1, 0]) * 2) / h_eta**2
        
        S[:, -1] = 2 * r_s[:, -1] / h_eta**2
        N[:, -1] = 0
        B[:, -1] = -(W[:, -1] + E[:, -1] + S[:, -1])
        F[1:-1, -1] = (r_e[1:-1, -1]*(phi[2:, -1] - phi[1:-1, -1]) - r_w[1:-1, -1]*(phi[1:-1, -1] - phi[:-2, -1])) / h_xi**2 + \
                       (r_s[1:-1, -1]*(phi[1:-1, -2] - phi[1:-1, -1]) * 2) / h_eta**2
        
        # Углы (i=0, j=0 и i=0, j=N_eta)
        F[0, 0] = (r_e[0, 0]*(phi[1, 0] - phi[0, 0]) * 2) / h_xi**2 + \
                  (r_n[0, 0]*(phi[0, 1] - phi[0, 0]) * 2) / h_eta**2
        F[0, -1] = (r_e[0, -1]*(phi[1, -1] - phi[0, -1]) * 2) / h_xi**2 + \
                   (r_s[0, -1]*(phi[0, -2] - phi[0, -1]) * 2) / h_eta**2
        
        # 3. Внешняя граница (i=N_xi, Дирихле => dphi = 0)
        B[-1, :] = 1.0
        W[-1, :] = 0.0; E[-1, :] = 0.0; S[-1, :] = 0.0; N[-1, :] = 0.0
        F[-1, :] = 0.0
        
        solver.factorize(S, W, B, E, N, alpha_stone)
        dphi = solver.solve(F)
        
        phi += dphi
        err = np.max(np.abs(dphi))
        if err < eps:
            print(f"M={M_inf} (SIP) сошлось за {it} итераций (err={err:.2e})")
            break
    return phi

In [15]:
machs = [0.1, 0.2, 0.3, 0.4]
results = {}
phi_current = None

for M in machs:
    phi_current = solve_for_mach_sip(M, phi_current)
    results[M] = phi_current

table_data = []
idx_90 = np.argmin(np.abs(eta - np.pi/2))

for M in machs:
    phi = results[M]
    phi_eta = np.zeros(N_eta + 1)
    phi_eta[1:N_eta] = (phi[0, 2:N_eta+1] - phi[0, 0:N_eta-1]) / (2 * h_eta)
    phi_eta[0] = 0; phi_eta[N_eta] = 0
    
    q2 = phi_eta**2
    q = np.sqrt(q2)
    
    if M == 0:
        cp = 1 - q2
        m_loc = np.zeros_like(q2)
    else:
        rho = (1 + 0.5 * (gamma - 1) * M**2 * (1 - q2))**(1/(gamma-1))
        cp = 2 / (gamma * M**2) * (rho**gamma - 1)
        a2 = 1 + 0.5 * (gamma - 1) * M**2 * (1 - q2)
        m_loc = q * M / np.sqrt(a2)
    
    df_m = pd.DataFrame({'eta_deg': eta * 180 / np.pi, 'q': q, 'Cp': cp, 'M_loc': m_loc})
    df_m.to_csv(f'results_sip_M_{M}.csv', index=False)
    
    table_data.append({'M_inf': M, 'q_pi/2': q[idx_90], 'Cp_pi/2': cp[idx_90], 'M_loc_pi/2': m_loc[idx_90]})

df_results = pd.DataFrame(table_data)
print("\nРезультаты (SIP) в точке pi/2:")
print(df_results.to_string(index=False))

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
for M in machs:
    plt.plot(eta*180/np.pi, pd.read_csv(f'results_sip_M_{M}.csv')['Cp'], label=f'M={M}')
plt.title('Коэффициент давления Cp (SIP)')
plt.xlabel('eta (deg)'); plt.ylabel('Cp'); plt.legend(); plt.grid()

plt.subplot(1, 2, 2)
for M in machs:
    if M == 0: continue
    plt.plot(eta*180/np.pi, pd.read_csv(f'results_sip_M_{M}.csv')['M_loc'], label=f'M={M}')
plt.title('Локальное число Маха M_loc (SIP)')
plt.xlabel('eta (deg)'); plt.ylabel('M_loc'); plt.legend(); plt.grid()
plt.show()

C:\Users\Ivan\AppData\Local\Temp\ipykernel_11604\3840432911.py:26: RuntimeWarning: invalid value encountered in scalar divide
  self.e[i, j] = (E[i, j] - alpha * self.c[i, j] * f_prev) / self.d[i, j]
C:\Users\Ivan\AppData\Local\Temp\ipykernel_11604\3840432911.py:27: RuntimeWarning: invalid value encountered in scalar divide
  self.f[i, j] = (N[i, j] - alpha * self.b[i, j] * e_prev) / self.d[i, j]
C:\Users\Ivan\AppData\Local\Temp\ipykernel_11604\3840432911.py:36: RuntimeWarning: invalid value encountered in scalar divide
  y[i, j] = (res[i, j] - self.b[i, j] * y_s - self.c[i, j] * y_w) / self.d[i, j]


KeyboardInterrupt: 